[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geraldmc/irilab2026/blob/main/notebooks/r1/r1-q4/01_integration.ipynb)

# R1-Q4 Week 2 — Integrate Wang 2023 with AtGenExpress

### This notebook produces `integrated_matrix.parquet`, the input the classifier trains on next week.

This notebook integrates Wang 2023 cold-stress RNA-seq data with the AtGenExpress microarray dataset so that a classifier can be trained on one and tested on the other. The integrated matrix is the input to the classifier you train in Week 3, and the integration diagnostics will go into your Week 2 EQ#2 report.

By the end of this notebook you will have:

- Wang 2023 RNA-seq counts VST-transformed and aligned to ATH1 microarray probe space (`wang_aligned.parquet`).
- A ComBat-corrected joint expression matrix combining AtGenExpress and Wang 2023 (`integrated_matrix.parquet`), ready as the training input in Week 3.
- An `integration_summary.json` recording sample counts, gene counts after alignment, ComBat parameters, and before/after batch-effect diagnostics, ready as Week 2 EQ#2 input.

In [ ]:
# Cell 0 — probe
try:
    import irilab2026 as iri
    print(f"irilab2026 {iri.__version__} already installed.")
    print("If you want to pull the latest changes, run Cell 1 below.")
except ModuleNotFoundError:
    print("irilab2026 not installed — run Cell 1 below.")

In [ ]:
# Cell 1 — install
# First run in a fresh Colab session: run this cell.
# Later runs in the same session: skip this cell.
!pip install git+https://github.com/geraldmc/irilab2026.git@main

**A note about a possible restart prompt.** If Colab shows a "RESTART SESSION" banner after the install: click it, wait for the kernel to reconnect, then continue from the next cell. The install does not need to be run again. If no banner appears, ignore this and move on.

In [ ]:
# Cell 2 — mount and setup
# Mount Drive, set up irilab2026, point to the R1-Q4 output directory.
import irilab2026 as iri
iri.setup()

OUTPUT_DIR = iri.output_dir("r1_q4")
print(f"Output directory: {OUTPUT_DIR}")

### 1) Defensive load of `precommit.json` (the scope, 168 h, and null decisions drive everything downstream).

You committed to three decisions in Notebook 00 and wrote them to `precommit.json`. This notebook reads that file before doing anything else.

The cell below does two things:

1. **Loads** `precommit.json` from the notebook directory.
2. **Validates** that the three top-level keys are present and that each primary field holds one of the values you were allowed to choose.

If the file is missing or a value is unexpected, the cell raises an error and points you back to Notebook 00. Fix the problem there, re-run Notebook 00's closeout cell to regenerate `precommit.json`, then come back and re-run this cell.

The validated dict is bound to `precommit` and used by every later section.

In [ ]:
import json
from pathlib import Path

PRECOMMIT_PATH = Path("precommit.json")

# Expected structure: top-level key -> (primary field name, allowed values)
EXPECTED = {
    "data_source_and_scope": ("source", {"GEO", "TAIR_NASCArrays"}),
    "wang_168h_handling":    ("decision", {"exclude", "include_as_ood"}),
    "above_chance_null":     ("decision", {"binary_cold_vs_control", "multiclass_behavior"}),
}

if not PRECOMMIT_PATH.exists():
    raise FileNotFoundError(
        f"{PRECOMMIT_PATH} not found. "
        "Run Notebook 00 to its closeout cell, then re-run this cell."
    )

with PRECOMMIT_PATH.open() as f:
    precommit = json.load(f)

# Check top-level keys.
missing = [k for k in EXPECTED if k not in precommit]
if missing:
    raise KeyError(
        f"precommit.json is missing top-level key(s): {missing}. "
        "Re-run Notebook 00's closeout cell to regenerate the file."
    )

# Check primary field on each top-level key.
for top_key, (field, allowed) in EXPECTED.items():
    block = precommit[top_key]
    if field not in block:
        raise KeyError(
            f"precommit['{top_key}'] is missing field '{field}'. "
            "Re-run Notebook 00's closeout cell."
        )
    value = block[field]
    if value not in allowed:
        raise ValueError(
            f"precommit['{top_key}']['{field}'] = {value!r}, "
            f"but expected one of {sorted(allowed)}. "
            "Revisit the relevant Precommit section in Notebook 00."
        )

print("precommit.json loaded and validated.")
for top_key, (field, _) in EXPECTED.items():
    print(f"  {top_key}.{field} = {precommit[top_key][field]!r}")

**What just happened:** `precommit.json` is now loaded as the `precommit` dict, and the three decisions you committed to are printed above. Every later section in this notebook branches on these values:

- `precommit["data_source_and_scope"]["source"]` decides which loader runs in Section 2.
- `precommit["wang_168h_handling"]["decision"]` decides the 168 h filter in Section 3.
- `precommit["above_chance_null"]["decision"]` is read by Notebook 03, not here — but Section 1 still validates it so a bad value surfaces before the long VST and ComBat steps, not after.

**If you need to change a decision:** edit the relevant Precommit section in Notebook 00, re-run its closeout cell to overwrite `precommit.json`, and re-run this notebook from the top.

### 2) Load AtGenExpress per the precommitted scope; filter the training set to shoot samples only (per Considerations)

#### 2.1 Load AtGenExpress (branches on your data source decision)

AtGenExpress is hosted in two places that matter for this question:

- **GEO** (GSE accessions, accessed via `GEOparse`)
- **TAIR / NASCArrays** (flat files, accessed via the mentor-provided loader)

In Notebook 00 you committed to one of these in `precommit["data_source_and_scope"]["source"]`. The cell below reads that decision and loads from the matching source. Both paths return the same thing: an expression dataframe `expr` (genes × samples) and a metadata dataframe `meta` (samples × annotations).

The expression matrix is already VST-normalized for GEO and log2-normalized for TAIR/NASCArrays — both are on a log scale and roughly comparable, but the alignment work in Section 5 assumes both inputs are on a log scale, not raw counts. If you ever switch data sources mid-project, re-run from Notebook 00.

In [ ]:
from r1q4_helpers import load_atgenexpress_geo, load_atgenexpress_nascarrays

source = precommit["data_source_and_scope"]["source"]

if source == "GEO":
    expr, meta = load_atgenexpress_geo()
elif source == "TAIR_NASCArrays":
    expr, meta = load_atgenexpress_nascarrays()
else:
    # Section 1 already validated this; defensive belt-and-suspenders.
    raise ValueError(f"Unexpected source: {source!r}")

print(f"Loaded AtGenExpress from {source}.")
print(f"  expr: {expr.shape[0]:,} genes × {expr.shape[1]:,} samples")
print(f"  meta: {meta.shape[0]:,} samples × {meta.shape[1]} annotation columns")
print(f"  meta columns: {list(meta.columns)}")

**What to notice in the printout above:**

- **Sample count.** AtGenExpress is large (hundreds of samples). The exact number depends on which source you chose; both paths cover the same underlying study but GEO and TAIR/NASCArrays curate differently.
- **Metadata columns.** Look for the column that names the tissue (`tissue`, `organ`, or similar — depends on source). You will use it in 2.2. Also look for the column naming the stress treatment; you will use it in Section 7's diagnostics.
- **Gene count.** Should be in the low tens of thousands. AtGenExpress is ATH1 microarray — about 22,000 probes mapped to Arabidopsis genes. If the count is dramatically lower, the load returned a filtered subset and something is wrong; stop and re-load.

The `expr` and `meta` dataframes are now in scope. Don't overwrite them.

#### 2.2 Filter to shoot samples

R1-Q4 is a question about **shoot stress responses**. AtGenExpress contains samples from multiple tissues (root, shoot, whole seedling, etc.); for this question, only shoot samples are in scope. Filtering happens here, once, before any normalization or integration work.

Use the tissue column you spotted in 2.1's printout. The exact column name and the exact label for "shoot" depend on which source you loaded from — inspect `meta` if you need to confirm both before writing the filter.

**What to produce:**

- `meta_shoot`: rows of `meta` where the tissue is shoot.
- `expr_shoot`: columns of `expr` matching `meta_shoot`'s sample IDs, in the same order.

The sample IDs are the index of `meta` and the column names of `expr`. Keep them aligned — `expr_shoot.columns` and `meta_shoot.index` must match exactly after the filter.

In [ ]:
# Practice 2.2 — your code here.
#
# Goal: build meta_shoot and expr_shoot, with aligned sample IDs.
#
# Hint: inspect meta to find the tissue column and the exact shoot label before
# writing the filter. The label is not necessarily lowercase "shoot".

# meta_shoot = ...
# expr_shoot = ...

# Sanity prints — uncomment after you've built both variables.
# print(f"meta_shoot: {meta_shoot.shape[0]:,} shoot samples")
# print(f"expr_shoot: {expr_shoot.shape[1]:,} sample columns")
# assert list(expr_shoot.columns) == list(meta_shoot.index), "sample IDs misaligned"

**Sanity-check before continuing:** the `assert` in the cell above must pass. If `expr_shoot.columns` and `meta_shoot.index` are in different orders (or contain different IDs), every downstream operation that crosses expression and metadata — alignment in Section 5, ComBat in Section 6, the diagnostic plots in Section 7 — will silently mismatch samples.

**What to notice:**

- **Sample count after filter.** AtGenExpress shoot coverage is the larger share of the original sample count, but not all of it. Roots and whole seedlings drop out here.
- **Stress treatments still in scope.** Print `meta_shoot["<treatment_column>"].value_counts()` if you want to confirm cold-stress samples survived the tissue filter. If they didn't, stop — something is wrong with the filter.

`expr_shoot` and `meta_shoot` are what Section 3 onward operates on. The original `expr` and `meta` stay in scope but are no longer used.

### 3) Load Wang 2023; filter timepoints per the precommitted 168 h handling

#### 3.1 Load Wang 2023 (cold RNA-seq)

Wang 2023 is an RNA-seq study of Arabidopsis shoot responses to cold stress across multiple timepoints. It is the second of the two data sources you are integrating in this notebook.

The cell below loads the Wang 2023 expression matrix and metadata via the mentor-provided helper. The return shape mirrors Section 2: an expression dataframe `expr_wang` (genes × samples) and a metadata dataframe `meta_wang` (samples × annotations).

Important: Wang 2023 ships as **raw counts**, not log-normalized values. AtGenExpress from Section 2 is already on a log scale. The two are not directly comparable until Section 4 applies VST to Wang. Don't try to integrate or align before that.

In [ ]:
from r1q4_helpers import load_wang2023

expr_wang, meta_wang = load_wang2023()

print(f"Loaded Wang 2023.")
print(f"  expr_wang: {expr_wang.shape[0]:,} genes × {expr_wang.shape[1]:,} samples")
print(f"  meta_wang: {meta_wang.shape[0]:,} samples × {meta_wang.shape[1]} annotation columns")
print(f"  meta_wang columns: {list(meta_wang.columns)}")

**What to notice in the printout above:**

- **Sample count.** Wang 2023 is much smaller than AtGenExpress — tens of samples, not hundreds. This asymmetry is the reason ComBat in Section 6 matters: a naïve concatenation would let AtGenExpress dominate any joint analysis.
- **Timepoint column.** Look for the column naming the time-after-cold-treatment for each sample. The values are in hours (e.g., 1, 3, 6, 24, 168). You will use this column in 3.2.
- **Gene count.** Wang 2023 covers more genes than the ATH1 microarray because RNA-seq is unbiased — but only the genes that overlap with AtGenExpress's probes will survive Section 5's alignment. Most of the extra coverage drops out there.

The `expr_wang` and `meta_wang` dataframes are now in scope. Don't overwrite them.

### 3.2 Apply your 168 h timepoint decision

In Notebook 00 you committed to one of two ways of handling the Wang 2023 168 h timepoint:

- `"exclude"` — drop 168 h samples from the dataset entirely. They do not pass through integration, the classifier, or evaluation.
- `"include_as_ood"` — keep 168 h samples in the dataset but tag them as out-of-distribution. They pass through integration here, but Notebook 02 excludes them from classifier training and Notebook 03 evaluates the trained classifier on them separately to test whether early cold-response signatures still hold at the late timepoint.

The cell below reads your decision from `precommit["wang_168h_handling"]["decision"]` and branches. In both paths, the goal is to produce a single pair of dataframes that flow into Section 4:

- `meta_wang` (potentially with rows dropped and/or an added `ood` column)
- `expr_wang` (columns matching `meta_wang.index`, in the same order)

The `ood` column is a boolean: `True` for held-out samples, `False` for in-distribution samples. For the `"exclude"` path, the column is added and set to `False` for all surviving rows — so downstream code can always read `meta_wang["ood"]` regardless of which path ran.

First, inspect `meta_wang` to find the timepoint column name and confirm the 168 h label. Then write the branch.

In [ ]:
# Practice 3.2 — your code here.
#
# Goal: apply the 168 h decision from precommit, producing meta_wang and expr_wang
# with aligned sample IDs. Add an `ood` column to meta_wang regardless of path.
#
# Hint: the timepoint column may be named "time", "time_hours", "timepoint", or
# similar — inspect meta_wang first. The 168 h label is most likely the integer
# 168 but could be a string like "168h"; check before writing the comparison.

decision = precommit["wang_168h_handling"]["decision"]

if decision == "exclude":
    # Drop 168 h rows from meta_wang; drop matching columns from expr_wang;
    # add ood column set to False everywhere.
    raise NotImplementedError("Practice 3.2 — write the 'exclude' branch.")

elif decision == "include_as_ood":
    # Keep all rows; add ood column set to True for 168 h rows, False otherwise.
    # expr_wang is unchanged on this path.
    raise NotImplementedError("Practice 3.2 — write the 'include_as_ood' branch.")

else:
    # Section 1 already validated this; defensive belt-and-suspenders.
    raise ValueError(f"Unexpected decision: {decision!r}")

# Sanity prints — uncomment after both branches are written.
# print(f"Decision applied: {decision}")
# print(f"meta_wang: {meta_wang.shape[0]:,} samples ({meta_wang['ood'].sum()} tagged ood)")
# print(f"expr_wang: {expr_wang.shape[1]:,} sample columns")
# assert list(expr_wang.columns) == list(meta_wang.index), "sample IDs misaligned"

**Sanity-check before continuing:** the `assert` in the cell above must pass, and `meta_wang["ood"]` must exist as a boolean column. Notebook 02's classifier training reads this column directly.

**What to notice:**

- **Sample count after filter** depends on which branch ran. The `"exclude"` path drops 168 h samples; the `"include_as_ood"` path keeps all samples but tags 168 h as `ood=True`.
- **The `ood` column is present on both paths.** This is deliberate — downstream notebooks can always read `meta_wang["ood"]` without first checking which decision was made. On the `"exclude"` path, every surviving row has `ood=False`.
- **expr_wang and meta_wang are the in-scope objects going forward.** Section 4 applies VST to `expr_wang`. Sections 5 and 6 align and batch-correct against Section 2's `expr_shoot`.

**Forward pointer to Notebook 02 and Notebook 03:** the `ood` column is the contract. Notebook 02 filters training data on `meta_wang["ood"] == False`. Notebook 03 evaluates the trained classifier on held-out samples by reading the same column. If the column is wrong here, both downstream notebooks are wrong.

### 4) VST on Wang 2023 RNA-seq counts

Wang 2023 is on a raw-count scale; AtGenExpress is already on a log scale. To compare or integrate them in later sections, both need to be on the same kind of scale. **Variance-stabilizing transformation (VST)** is how RNA-seq counts get there.

What VST does, in one sentence: it transforms counts so that variance is roughly constant across the expression range, instead of the count-data default where high-expression genes have much higher variance than low-expression genes. The output is on a log-ish scale comparable to log2 microarray data.

The cell below runs VST via `pyDESeq2`. Three things are worth knowing before you read the code:

1. **`pyDESeq2` expects samples × genes**, the opposite of the genes × samples orientation we've been using. The cell transposes on the way in and transposes back on the way out.
2. **`use_design=False` (a "blind" VST).** The transformation doesn't see your stress labels. This is the right choice for cross-study integration — you don't want the cold/control labels influencing the variance fit, because that would smuggle the signal you're trying to learn into the normalization.
3. **VST needs integer counts.** Wang 2023 ships as raw counts, so this works directly. If the loader ever returns floats, the cell will fail loudly here, not silently downstream.

The output is `expr_wang_vst`, a genes × samples dataframe on a log-ish scale. The original `expr_wang` stays in scope for diagnostic comparison.

In [ ]:
from pydeseq2.dds import DeseqDataSet

# pyDESeq2 expects samples × genes; we've been working genes × samples. Transpose in.
counts_samples_by_genes = expr_wang.T

# Sanity: counts must be non-negative integers.
if not (counts_samples_by_genes.values >= 0).all():
    raise ValueError("expr_wang contains negative values; not raw counts.")
if not counts_samples_by_genes.dtypes.apply(lambda d: d.kind in "iu").all():
    raise ValueError("expr_wang is not integer-typed; VST expects raw counts.")

# Build the DeseqDataSet. design = "~1" via use_design=False below; we pass a
# placeholder design here that pyDESeq2 requires for construction but won't use.
dds = DeseqDataSet(
    counts=counts_samples_by_genes,
    metadata=meta_wang,
    design="~1",  # intercept-only; blind VST ignores any condition labels.
)

# Run VST. use_design=False means the dispersion fit uses only the intercept —
# no peeking at stress labels.
dds.vst(use_design=False)

# Pull the result out of the layers dict and transpose back to genes × samples
# to match the rest of the notebook's convention.
expr_wang_vst = dds.layers["vst_counts"].T

# Wrap as a DataFrame with the original gene and sample IDs.
import pandas as pd
expr_wang_vst = pd.DataFrame(
    expr_wang_vst,
    index=expr_wang.index,
    columns=expr_wang.columns,
)

print(f"VST applied to Wang 2023.")
print(f"  expr_wang_vst: {expr_wang_vst.shape[0]:,} genes × {expr_wang_vst.shape[1]:,} samples")
print()
print(f"Before VST (raw counts):")
print(f"  value range: [{expr_wang.values.min()}, {expr_wang.values.max()}]")
print(f"  median per-gene variance: {expr_wang.var(axis=1).median():,.1f}")
print()
print(f"After VST:")
print(f"  value range: [{expr_wang_vst.values.min():.2f}, {expr_wang_vst.values.max():.2f}]")
print(f"  median per-gene variance: {expr_wang_vst.var(axis=1).median():.3f}")
print()
print(f"AtGenExpress (for scale comparison):")
print(f"  value range: [{expr_shoot.values.min():.2f}, {expr_shoot.values.max():.2f}]")
print(f"  median per-gene variance: {expr_shoot.var(axis=1).median():.3f}")

**What to notice in the printout above:**

- **The value range collapsed.** Raw counts ran from 0 to tens of thousands; VST output runs from roughly 0 to about 20. That's the log-ish scale.
- **Per-gene variance is now roughly comparable across the expression range.** Before VST, high-expression genes had variance in the millions while low-expression genes had variance near zero. After VST, the spread is much smaller. That's the "stabilization" in variance-stabilizing transformation.
- **Scale matches AtGenExpress.** Both `expr_wang_vst` and `expr_shoot` should now have similar value ranges and similar median per-gene variance. If they don't — if `expr_wang_vst` is on a wildly different scale than `expr_shoot` — Section 5's alignment will produce garbage and Section 6's ComBat won't save it. Stop and investigate.

**What's still in scope:**

- `expr_wang_vst` (genes × samples, log-ish scale) — used in Sections 5, 6, 7.
- `expr_wang` (genes × samples, raw counts) — kept for diagnostic comparison; not used in integration.
- `meta_wang` — unchanged from Section 3.

Section 5 picks up `expr_wang_vst` and `expr_shoot` and aligns them on a common gene set — the probe-to-gene mapping work.

### 5) Align RNA-seq gene IDs to ATH1 probe space

### 5.1 Collapse AtGenExpress probes to genes (AGI)

AtGenExpress was profiled on the ATH1 microarray. Each row of `expr_shoot` is a **probe** — a short DNA sequence that hybridizes to a transcript. Wang 2023 is RNA-seq; each row of `expr_wang_vst` is already a **gene**, identified by its AGI locus identifier (`AT1G01010`, `AT2G35930`, etc.).

To put both datasets on the same vocabulary, the ATH1 probes need to be mapped to AGI loci. The mapping is many-to-one and not always clean:

- Most probes map to exactly one AGI (the usual case).
- Some AGIs are covered by multiple probes (technical replicates on the array).
- A handful of probes don't map to any current AGI (deprecated identifiers, ambiguous designs).

The cell below loads the mentor-supplied probe-to-AGI mapping table and collapses `expr_shoot` so each row is one AGI. Where multiple probes map to the same AGI, the cell takes the **mean** of their expression values per sample. Unmapped probes are dropped.

Mean across probes is the conventional choice. Max (most-responsive probe) and median (robust to outlier probes) are defensible alternatives — this notebook uses mean.

In [ ]:
from r1q4_helpers import load_ath1_probe_to_agi

probe_to_agi = load_ath1_probe_to_agi()  # DataFrame: index=probe_id, column 'agi'

print(f"Probe-to-AGI mapping: {len(probe_to_agi):,} probes")
print(f"  unique AGIs covered: {probe_to_agi['agi'].nunique():,}")
print(f"  probes with no AGI mapping: {probe_to_agi['agi'].isna().sum():,}")

# Restrict expr_shoot to probes that have an AGI mapping.
mapped_probes = probe_to_agi.dropna(subset=["agi"]).index
expr_shoot_mapped = expr_shoot.loc[expr_shoot.index.intersection(mapped_probes)]

# Attach the AGI column for groupby.
expr_shoot_with_agi = expr_shoot_mapped.copy()
expr_shoot_with_agi["agi"] = probe_to_agi.loc[expr_shoot_with_agi.index, "agi"]

# Collapse: mean across probes per AGI per sample.
expr_shoot_by_agi = expr_shoot_with_agi.groupby("agi").mean()

print()
print(f"Before collapse: {expr_shoot.shape[0]:,} probes × {expr_shoot.shape[1]:,} samples")
print(f"After collapse:  {expr_shoot_by_agi.shape[0]:,} AGIs × {expr_shoot_by_agi.shape[1]:,} samples")
print(f"  probes dropped (no AGI mapping): {expr_shoot.shape[0] - len(expr_shoot_with_agi):,}")
print(f"  AGIs covered by >1 probe: {(probe_to_agi.dropna(subset=['agi'])['agi'].value_counts() > 1).sum():,}")

**What to notice in the printout above:**

- **Probes dropped.** A small fraction of ATH1 probes don't map to current AGIs. Those rows are gone. If the drop count is unexpectedly large (more than a few percent), the mapping table is out of date or wrong — stop and check.
- **AGIs covered by more than one probe.** Several thousand AGIs typically have 2+ probes on ATH1. Mean across probes is what's collapsing those.
- **Final AGI count.** ATH1 covers about 22,000 Arabidopsis genes; the collapsed matrix should be in that range.

`expr_shoot_by_agi` is now genes (AGIs) × samples, comparable in shape and vocabulary to `expr_wang_vst`. Section 5.2 intersects the two on shared AGIs.

### 5.2 Intersect Wang and AtGenExpress on shared genes

`expr_shoot_by_agi` and `expr_wang_vst` are both indexed by AGI, but their gene sets don't fully overlap:

- AtGenExpress covers only the AGIs probed on ATH1 (a ~2003 design — misses some genes added to TAIR later).
- Wang 2023, being RNA-seq, covers the full annotated genome — including many genes ATH1 never had probes for.

Integration requires the same genes in both matrices, in the same order. The goal of this section is to produce two aligned matrices:

- `expr_shoot_aligned` (AGIs × AtGenExpress samples)
- `expr_wang_aligned` (AGIs × Wang samples)

Both on the same AGI index, in the same order. After this, the two matrices can sit side by side: same rows, different columns.

**The hazard.** Misaligned matrices fail silently. If `expr_shoot_aligned` and `expr_wang_aligned` end up on slightly different AGI orderings — even one swap — every gene-wise operation downstream of here is wrong about which gene it's looking at. Section 6's ComBat won't catch it. The diagnostic plots in Section 7 won't catch it. The classifier in Notebook 02 will train on jumbled gene identities and produce confident, meaningless predictions.

The practice cell below has you do the alignment and includes an assertion that the two indices match exactly. The assertion is the load-bearing check.

In [ ]:
# Practice 5.2 — your code here.
#
# Goal: produce expr_shoot_aligned and expr_wang_aligned, both indexed by the
# same set of AGIs in the same order.
#
# Hint: find the intersection of the two AGI sets, then use .loc[] with that
# intersection on both matrices. Order matters — make sure both matrices use
# the same ordering (e.g., sorted, or the order from one of the originals).

# common_agis = ...
# expr_shoot_aligned = ...
# expr_wang_aligned = ...

# Sanity prints and assertions — uncomment after you've built both matrices.
# print(f"Shared AGIs: {len(common_agis):,}")
# print(f"expr_shoot_aligned: {expr_shoot_aligned.shape[0]:,} genes × {expr_shoot_aligned.shape[1]:,} samples")
# print(f"expr_wang_aligned:  {expr_wang_aligned.shape[0]:,} genes × {expr_wang_aligned.shape[1]:,} samples")
# assert list(expr_shoot_aligned.index) == list(expr_wang_aligned.index), "AGI indices misaligned"
# assert expr_shoot_aligned.shape[0] == len(common_agis), "expr_shoot_aligned has wrong number of genes"
# assert expr_wang_aligned.shape[0] == len(common_agis), "expr_wang_aligned has wrong number of genes"

**Sanity-check before continuing:** all three `assert`s in the cell above must pass. If any fails, the matrices are not aligned and nothing downstream of here is trustworthy.

**What to notice:**

- **Shared AGI count.** Should be in the high teens of thousands — most of ATH1 is covered by Wang 2023's RNA-seq. The genes dropped here are ATH1 probes for AGIs no longer in the current annotation, plus Wang's extra genes that ATH1 never covered.
- **Sample counts unchanged.** Section 5 collapses and intersects *rows* (genes); it doesn't touch *columns* (samples). `expr_shoot_aligned.shape[1]` should match `meta_shoot.shape[0]`, and `expr_wang_aligned.shape[1]` should match `meta_wang.shape[0]`.

**What's in scope going forward:**

- `expr_shoot_aligned`, `expr_wang_aligned` — the integration inputs.
- `meta_shoot`, `meta_wang` — sample-level metadata, unchanged.
- The earlier matrices (`expr_shoot`, `expr_shoot_by_agi`, `expr_wang`, `expr_wang_vst`) are still in memory but no longer used. Section 6's ComBat operates on the `_aligned` pair.

Section 6 stacks these two matrices horizontally and applies ComBat to correct the batch effect between studies.

### 6) ComBat across both datasets, with platform as the batch variable (VST first, then ComBat — the order matters)

After Section 5, you have two matrices on the same gene set: `expr_shoot_aligned` (AtGenExpress) and `expr_wang_aligned` (Wang 2023). They share rows but they came from different studies, on different platforms, processed by different labs.

If you concatenate them as-is and train a classifier, the classifier will mostly learn **which study a sample came from** — not what stress condition it was under. The platform difference is large and the stress signal is small; without correction, the platform difference dominates.

**ComBat** is the standard tool for this. It models each gene's expression as a sum of (true biological signal) + (batch effect) + (noise), then estimates the batch effect per gene per study and subtracts it. The mathematical framework is empirical Bayes; the practical effect is "remove the study-level offset and scale, leave the biology."

The cell below runs ComBat via `inmoose.pycombat_norm`. Four things are worth knowing before you read the code:

1. **`pycombat_norm` is the right call for our case** (VST-normalized, log-scale data). `pycombat_seq` is for raw counts — we already did VST in Section 4.
2. **Genes × samples orientation** is what `pycombat_norm` expects. No transpose needed.
3. **The `mod` argument preserves biological signal.** Without it, ComBat may flatten the stress-vs-control distinction that the classifier needs to learn. We pass a stress-vs-control covariate via `mod`, telling ComBat "remove the study effect but keep the stress effect intact."
4. **The stress-vs-control labels are harmonized across studies** via a mentor helper. AtGenExpress and Wang 2023 name their treatments differently; the helper maps both to a unified `"control"`/`"stress"` label.

The outputs are `expr_shoot_combat` and `expr_wang_combat` — same shapes as their `_aligned` inputs, but on a common batch-corrected scale.

In [ ]:
from inmoose.pycombat import pycombat_norm
from r1q4_helpers import harmonize_stress_label
import pandas as pd
import numpy as np

# Stack the two matrices horizontally: same genes, all samples together.
# AtGenExpress columns first, then Wang. The batch and mod arrays must follow
# this same ordering.
expr_combined = pd.concat([expr_shoot_aligned, expr_wang_aligned], axis=1)

# Build the batch label vector: one entry per sample column.
batch = (
    ["atgenexpress"] * expr_shoot_aligned.shape[1]
    + ["wang"]        * expr_wang_aligned.shape[1]
)

# Build the harmonized multi-class stress label across both studies.
# Returns a Series indexed by sample ID, e.g. "control", "cold_stress",
# "heat_stress", "drought_stress", "uv_b_stress", ...
stress_label = harmonize_stress_label(meta_shoot, meta_wang)
# Order must match expr_combined's column order.
stress_label = stress_label.loc[expr_combined.columns]

# Build the mod design matrix from the multi-class label. We pin "control"
# as the first category so get_dummies(drop_first=True) drops control and
# uses it as the implicit baseline; the remaining columns are stress-class
# indicators (1 = this class, 0 = not this class).
class_order = ["control"] + sorted(set(stress_label) - {"control"})
stress_label = stress_label.astype(pd.CategoricalDtype(categories=class_order, ordered=False))
mod = pd.get_dummies(stress_label, drop_first=True).astype(int)

print(f"ComBat inputs:")
print(f"  expr_combined: {expr_combined.shape[0]:,} genes x {expr_combined.shape[1]:,} samples")
print(f"  batch counts:  {pd.Series(batch).value_counts().to_dict()}")
print(f"  class counts:  {stress_label.value_counts().to_dict()}")
print(f"  mod columns:   {list(mod.columns)}")
print()

# Cross-tabulate class by batch - confirms the design matrix structure
# (which classes appear in which batches, where the overlap is).
batch_series = pd.Series(batch, index=expr_combined.columns, name="batch")
class_by_batch = pd.crosstab(stress_label, batch_series)
print(f"Sample counts by class and batch:")
print(class_by_batch.to_string())
print()

# Run ComBat. This is the slow step - usually 30s to a few minutes.
expr_corrected = pycombat_norm(
    data=expr_combined,
    batch=batch,
    mod=mod,
)

# Split the corrected matrix back into per-study pieces, preserving the original
# column orderings of expr_shoot_aligned and expr_wang_aligned.
expr_shoot_combat = expr_corrected.loc[:, expr_shoot_aligned.columns]
expr_wang_combat  = expr_corrected.loc[:, expr_wang_aligned.columns]

print(f"ComBat outputs:")
print(f"  expr_shoot_combat: {expr_shoot_combat.shape[0]:,} genes x {expr_shoot_combat.shape[1]:,} samples")
print(f"  expr_wang_combat:  {expr_wang_combat.shape[0]:,} genes x {expr_wang_combat.shape[1]:,} samples")
print()
print(f"Per-study mean expression (before -> after ComBat):")
print(f"  AtGenExpress: {expr_shoot_aligned.values.mean():.2f} -> {expr_shoot_combat.values.mean():.2f}")
print(f"  Wang 2023:    {expr_wang_aligned.values.mean():.2f} -> {expr_wang_combat.values.mean():.2f}")
print(f"  Difference:   {abs(expr_shoot_aligned.values.mean() - expr_wang_aligned.values.mean()):.2f}"
      f" -> {abs(expr_shoot_combat.values.mean() - expr_wang_combat.values.mean()):.2f}")

**What to notice in the printout above:**

- **Per-study mean expression converged.** Before ComBat, the two studies likely had visibly different mean expression levels (different platforms, different normalization choices). After ComBat, the means should be close. If the gap actually got *larger*, something is wrong — stop and check the batch labels.
- **Shapes are preserved.** `expr_shoot_combat` has the same shape as `expr_shoot_aligned`; `expr_wang_combat` matches `expr_wang_aligned`. ComBat only changes values, not dimensions.
- The `mod` matrix has **one column per stress class** minus the control baseline. AtGenExpress covers eight or nine stress conditions depending on your data-source precommit (GEO vs. TAIR/NASCArrays), so expect either eight or nine columns. If you see fewer columns than that, or any all-zero columns, the harmonization helper missed some labels and the correction may have erased class-specific signal.
- The class-by-batch crosstab shows two rows shared across batches (`control` and `cold_stress`) and the remaining classes populated only by AtGenExpress. That's the expected shape: the shared rows anchor the batch-effect estimate, the AtGenExpress-only rows ride along with the per-batch shift that the shared rows establish.

**A point about ComBat's guarantees.** ComBat removes the *modeled* batch effect — the linear shift and scale difference between studies. It does not remove platform-specific gene response patterns (genes that behave fundamentally differently on RNA-seq vs microarray). If a gene's true biology differs by detection method, no amount of batch correction will reconcile it. Section 7 looks for cases where this matters.

**What's in scope going forward:**

- `expr_shoot_combat`, `expr_wang_combat` — the batch-corrected matrices.
- `meta_shoot`, `meta_wang` — unchanged.
- The `_aligned` matrices stay in memory for diagnostic comparison in Section 7 (before vs after ComBat).

Section 7 runs integration diagnostics: PCA before vs after correction, mean-difference plots, and a check that stress-vs-control signal survived the correction.

### 7) Integration diagnostics (sample counts; per-class counts; gene counts after alignment; PCA before/after)

### 7.1 PCA before and after ComBat

The first diagnostic question: **after ComBat, do samples still cluster by study, or do they mix?**

PCA gives a fast visual answer. On the pre-ComBat data, the largest source of variation is almost always the study itself — so the first two principal components separate AtGenExpress from Wang into two clouds. After ComBat, if correction worked, those clouds should overlap. If they don't, ComBat didn't actually do anything useful (or did something wrong).

You will write the PCA and the plot. Side-by-side: pre-ComBat on the left, post-ComBat on the right. Both colored by study.

**Inputs available:**

- `expr_combined` from Section 6 — the pre-ComBat stacked matrix (genes × all samples).
- `expr_corrected` from Section 6 — the post-ComBat version.
- `batch` from Section 6 — the per-sample study label list.

**Hint:** `sklearn.decomposition.PCA` wants samples × features, so transpose each matrix before fitting. Fit and transform on each separately (don't reuse one PCA's components for the other; that defeats the comparison). Use `matplotlib.pyplot.subplots(1, 2)` for the side-by-side layout.

In [ ]:
# Practice 7.1 — your code here.
#
# Goal: produce a side-by-side scatter of PC1 vs PC2, pre-ComBat (left) and
# post-ComBat (right). Color points by study (AtGenExpress vs Wang).
#
# Hint: PCA wants samples × features, so transpose. Fit and transform each
# matrix separately; don't reuse one PCA across both.

from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# pca_before = ...
# pcs_before = ...
# pca_after = ...
# pcs_after = ...

# fig, axes = plt.subplots(1, 2, figsize=(12, 5))
# ... colored scatter on axes[0] (before) and axes[1] (after) ...
# plt.show()

**What to notice in the plots:**

- **Before ComBat (left).** AtGenExpress and Wang samples should occupy clearly separate regions of the PC1/PC2 plane. PC1 usually captures the study difference (often >50% of variance); the gap between the two clouds is the batch effect ComBat will try to remove.
- **After ComBat (right).** The two studies should overlap substantially. Perfect overlap isn't expected — the studies cover different stress treatments, different sample sizes, different timepoints — but the systematic separation should be gone.
- **What "didn't work" looks like.** If the two clouds are still cleanly separated after ComBat, something is wrong: bad batch labels, ComBat run on the wrong matrix, or the studies are too different to align linearly.

Move on to 7.2 only if the post-ComBat plot shows meaningful mixing. If it doesn't, stop and revisit Section 6.

### 7.2 Did the stress signal survive?

ComBat with `mod` is supposed to preserve biological signal while removing the study effect. 7.1 checked the study effect was removed. This subsection checks the biological signal survived.

The diagnostic: for each study separately, compute **per-gene mean expression in stress samples minus mean expression in control samples**, both before and after ComBat. Then plot before vs after, gene by gene, with the y = x diagonal drawn.

- **Points on the diagonal** → the stress-vs-control difference for that gene is unchanged. Signal preserved.
- **Points pulled toward zero (flattened to the horizontal axis)** → ComBat erased the stress difference for that gene. Signal lost.
- **Points scattered randomly** → noise dominates; the gene wasn't reliably differential to begin with.

The cell below runs the calculation and produces one panel per study. It also prints the Pearson correlation between the before and after vectors — a single number summarizing how well the signal was preserved across all genes. Correlation above about 0.8 is reassuring; below that, ComBat may have done more than just batch correction.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def stress_minus_control(expr, meta):
    """Per-gene mean(stress) - mean(control) for one study."""
    stress_samples  = meta.index[meta["stress_status"] == "stress"]
    control_samples = meta.index[meta["stress_status"] == "control"]
    return expr.loc[:, stress_samples].mean(axis=1) - expr.loc[:, control_samples].mean(axis=1)

# Attach harmonized stress status to both meta frames for the helper above.
# stress_status was computed in Section 6.
meta_shoot = meta_shoot.assign(stress_status=stress_status.loc[meta_shoot.index].values)
meta_wang  = meta_wang.assign(stress_status=stress_status.loc[meta_wang.index].values)

# Per-gene stress-minus-control, before and after, for each study.
diff_shoot_before = stress_minus_control(expr_shoot_aligned, meta_shoot)
diff_shoot_after  = stress_minus_control(expr_shoot_combat,  meta_shoot)
diff_wang_before  = stress_minus_control(expr_wang_aligned,  meta_wang)
diff_wang_after   = stress_minus_control(expr_wang_combat,   meta_wang)

# Correlations.
r_shoot = np.corrcoef(diff_shoot_before, diff_shoot_after)[0, 1]
r_wang  = np.corrcoef(diff_wang_before,  diff_wang_after)[0, 1]
print(f"Pearson r (before vs after stress-minus-control per gene):")
print(f"  AtGenExpress: r = {r_shoot:.3f}")
print(f"  Wang 2023:    r = {r_wang:.3f}")

# Side-by-side scatter with y=x diagonal.
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, before, after, title, r in [
    (axes[0], diff_shoot_before, diff_shoot_after, "AtGenExpress", r_shoot),
    (axes[1], diff_wang_before,  diff_wang_after,  "Wang 2023",    r_wang),
]:
    ax.scatter(before, after, s=4, alpha=0.3)
    lo = min(before.min(), after.min())
    hi = max(before.max(), after.max())
    ax.plot([lo, hi], [lo, hi], "k--", linewidth=1)
    ax.set_xlabel("stress - control (before ComBat)")
    ax.set_ylabel("stress - control (after ComBat)")
    ax.set_title(f"{title}  (r = {r:.3f})")

plt.tight_layout()
plt.show()

**What to notice in the plots:**

- **Most points should track the diagonal.** That means the stress-vs-control difference per gene was preserved by ComBat. The bulk of the cloud sitting on y = x is the "biology survived" picture.
- **Most points should land near the origin.** That just means most genes aren't differential between stress and control — that's normal. Only a minority of genes respond strongly to stress; the rest cluster near zero in both axes.
- **Points pulled horizontally toward zero** are the warning sign. If a gene had a large pre-ComBat stress-vs-control difference but a near-zero post-ComBat difference, ComBat flattened a real biological signal for that gene. A few such points are expected; many of them mean `mod` didn't do its job.
- **Pearson r per study** quantifies the picture. Above 0.8: signal preserved. 0.6–0.8: partial flattening. Below 0.6: ComBat over-corrected.

If r is low for either study, the gate in Section 8 will catch it — but you'll want to know which study failed and why before you get there.

### 7.3 Cold-marker spot check

7.2 gave you the picture across all genes at once. This subsection zooms in on a handful of well-known cold-responsive genes and looks at their expression sample by sample. If ComBat preserved stress signal in aggregate but flattened it for cold markers specifically, that's a failure mode 7.2's global correlation could miss.

The marker set below is six genes from the CBF/DREB1 regulon and downstream cold-responsive genes — the most-characterized cold-response markers in Arabidopsis:

- **CBF1 / DREB1B** (AT4G25490)
- **CBF2 / DREB1C** (AT4G25470)
- **CBF3 / DREB1A** (AT4G25480)
- **COR15A** (AT2G42540)
- **RD29A / COR78** (AT5G52310)
- **KIN1** (AT5G15960)

All six should show clearly elevated expression in cold-treated samples relative to controls — in both studies, before and after ComBat. The plot below shows that comparison directly.

In [ ]:
import matplotlib.pyplot as plt

# Cold-responsive marker genes from the CBF regulon.
cold_markers = {
    "CBF1 (AT4G25490)":   "AT4G25490",
    "CBF2 (AT4G25470)":   "AT4G25470",
    "CBF3 (AT4G25480)":   "AT4G25480",
    "COR15A (AT2G42540)": "AT2G42540",
    "RD29A (AT5G52310)":  "AT5G52310",
    "KIN1 (AT5G15960)":   "AT5G15960",
}

# Check which markers are actually in the integrated gene set; warn if any dropped.
shared_genes = set(expr_shoot_combat.index)
missing = [name for name, agi in cold_markers.items() if agi not in shared_genes]
if missing:
    print(f"Note: {len(missing)} marker(s) not in the integrated gene set: {missing}")
    cold_markers = {n: a for n, a in cold_markers.items() if a in shared_genes}

# Build a small dataframe: rows = (study, condition, marker), values = mean expression.
rows = []
for name, agi in cold_markers.items():
    for expr, meta, study in [
        (expr_shoot_combat, meta_shoot, "AtGenExpress"),
        (expr_wang_combat,  meta_wang,  "Wang 2023"),
    ]:
        for cond in ["control", "stress"]:
            samples = meta.index[meta["stress_status"] == cond]
            rows.append({
                "marker":    name,
                "study":     study,
                "condition": cond,
                "expression": expr.loc[agi, samples].mean(),
            })

import pandas as pd
marker_df = pd.DataFrame(rows)

# Grouped bar chart: one bar per (study, condition), one cluster per marker.
fig, ax = plt.subplots(figsize=(12, 5))
markers_list = list(cold_markers.keys())
x = np.arange(len(markers_list))
width = 0.2

for i, (study, cond) in enumerate([
    ("AtGenExpress", "control"), ("AtGenExpress", "stress"),
    ("Wang 2023",   "control"), ("Wang 2023",   "stress"),
]):
    vals = [
        marker_df.query("marker == @m and study == @study and condition == @cond")["expression"].iloc[0]
        for m in markers_list
    ]
    ax.bar(x + i * width, vals, width, label=f"{study} ({cond})")

ax.set_xticks(x + 1.5 * width)
ax.set_xticklabels(markers_list, rotation=30, ha="right")
ax.set_ylabel("Mean expression (post-ComBat)")
ax.set_title("Cold-marker expression by study and condition")
ax.legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()

**What to notice in the plot:**

- **Stress bars taller than control bars, per study, per marker.** This is the load-bearing observation. All six markers are cold-responsive; both studies include cold treatments; the bars should reflect that. If stress and control bars are equal-height for a given marker, ComBat may have flattened the signal at that gene specifically.
- **Per-marker absolute values should be roughly comparable across studies.** ComBat should have put both studies on the same scale; for any given marker, AtGenExpress and Wang values shouldn't differ by orders of magnitude. (A modest difference is normal — sample composition differs, the studies aren't identical.)
- **A missing marker.** If the printout warned that a marker isn't in the integrated gene set, that AGI dropped out at Section 5 (no ATH1 probe or no overlap with Wang's coverage). The marker is then absent from the plot. One or two dropping out is normal; all of them missing means something went wrong upstream.

**Why these six.** All are canonical cold-response genes in Arabidopsis. CBF1/2/3 are the master transcription factors driving the cold acclimation pathway; COR15A, RD29A, and KIN1 are downstream targets in the CBF regulon. If a classifier trained on this integrated dataset can't distinguish stress from control using genes this responsive, no amount of model tuning will save it.

**What's in scope going forward:**

- `expr_shoot_combat`, `expr_wang_combat` — the corrected matrices, validated.
- `meta_shoot`, `meta_wang` — now with `stress_status` attached.
- The before/after diagnostics in this section are not used by later sections; they're for your judgment about whether to proceed.

Section 8 is the integration gate: a structured decision point where you certify "diagnostics pass, integration is good enough" before NB02 picks up the corrected matrices.

### 8) Integration gate: does the integration look reasonable? (the explicit "stop here if not" check)

A **gate** is a structured stop-and-judge point. Section 7 gave you three diagnostics. This section is where you decide whether they collectively certify that the integration is good enough to hand to the classifier in Notebook 02.

Gates aren't decisions about parameters or methods — those were the precommits in Notebook 00. Gates are decisions about **work products**: did the thing I just built pass its own quality checks? The verdict is on the artifact, not on a choice.

This gate has four parts. The first three are per-diagnostic verdicts based on Section 7; the fourth is the overall call.

- **PCA mixing.** Did the post-ComBat PCA show meaningful overlap between AtGenExpress and Wang samples? (Section 7.1)
- **Stress signal preservation.** Were the per-study Pearson r values above 0.8 in both studies? (Section 7.2)
- **Cold marker behavior.** Did all six markers show stress > control in both studies? (Section 7.3)
- **Overall.** Combining the above, do you certify the integrated dataset for downstream use?

Each part takes a `pass` / `fail` / `partial` verdict and a short rationale in your own words. The verdicts and rationales are written to `integration_gate.json`. Notebook 02's Section 1 reads that file before doing any work.

A `fail` on any of the three diagnostic parts means **stop**: investigate the failure, revise Section 6 (or earlier), re-run, re-evaluate. Don't proceed to Notebook 02 with a failed gate. A `partial` is a judgment call — write what you observed and why you're choosing to continue or stop.

EQ#2, submitted at the end of this notebook, is your overall verdict and rationale copied into Notion.

**Pass / fail / partial criteria for each diagnostic.**

**PCA mixing (Section 7.1):**

- *pass* — the two studies' post-ComBat clouds overlap substantially. PC1 no longer cleanly separates AtGenExpress from Wang.
- *partial* — overlap exists but a visible study-level gradient remains on PC1 or PC2.
- *fail* — the two clouds are still cleanly separated post-ComBat. ComBat didn't do its job.

**Stress signal preservation (Section 7.2):**

- *pass* — Pearson r > 0.8 for both AtGenExpress and Wang, between pre-ComBat and post-ComBat stress-minus-control vectors.
- *partial* — r is between 0.6 and 0.8 for one or both studies, or one study passes and the other doesn't.
- *fail* — r < 0.6 for either study. ComBat over-corrected and flattened the biological signal.

**Cold marker behavior (Section 7.3):**

- *pass* — all six markers (CBF1, CBF2, CBF3, COR15A, RD29A, KIN1) show stress > control in both studies. Magnitudes don't need to match across studies, but the direction must.
- *partial* — most markers pass but one or two have inverted or flat patterns in one study.
- *fail* — markers are systematically flat or inverted in either study, or several markers were dropped at the alignment step.

**Overall:**

- *pass* — all three diagnostics pass. Hand off to Notebook 02 with confidence.
- *partial* — at least one diagnostic was partial, none failed. Document what you observed and why you're choosing to continue.
- *fail* — at least one diagnostic failed. Stop here and revisit.

In [ ]:
# Build the integration gate verdict dict. Fill in `verdict` and `rationale`
# for each part. r_observed values are auto-pulled from Section 7.2's scope.

integration_gate = {
    "pca_mixing": {
        "verdict": "...",       # "pass" | "partial" | "fail"
        "rationale": "...",     # one or two sentences in your own words
    },
    "stress_signal_preservation": {
        "verdict": "...",
        "rationale": "...",
        "r_observed": {
            "atgenexpress": float(r_shoot),
            "wang": float(r_wang),
        },
    },
    "cold_marker_behavior": {
        "verdict": "...",
        "rationale": "...",
    },
    "overall": {
        "verdict": "...",       # this is the EQ#2 verdict
        "rationale": "...",     # this is the EQ#2 rationale
    },
}

# Validate before writing: every verdict must be one of the three allowed strings.
ALLOWED_VERDICTS = {"pass", "partial", "fail"}
for part_name, part in integration_gate.items():
    if part["verdict"] not in ALLOWED_VERDICTS:
        raise ValueError(
            f"integration_gate['{part_name}']['verdict'] = {part['verdict']!r}, "
            f"but must be one of {sorted(ALLOWED_VERDICTS)}."
        )
    if not part["rationale"] or part["rationale"] == "...":
        raise ValueError(
            f"integration_gate['{part_name}']['rationale'] is empty. "
            "Write a one- or two-sentence justification in your own words."
        )

# Stop hard if overall is fail — don't write the file.
if integration_gate["overall"]["verdict"] == "fail":
    raise RuntimeError(
        "Overall verdict is 'fail'. Do not proceed to Notebook 02. "
        "Investigate the failed diagnostic, revise Section 6 or earlier, "
        "and re-run the notebook before re-evaluating this gate."
    )

import json
from pathlib import Path

with Path("integration_gate.json").open("w") as f:
    json.dump(integration_gate, f, indent=2)

print("integration_gate.json written.")
print(f"Overall verdict: {integration_gate['overall']['verdict']}")

### 9) Closeout: save the integrated matrix and `integration_summary.json`; submit EQ#2

You've integrated AtGenExpress and Wang 2023, certified the result through the integration gate, and have an `integration_gate.json` that records the verdict. Two things remain:

1. **Save the integrated dataset to disk.** Notebook 02 will load these files in its own Section 1 (defensive load, same pattern as this notebook's Section 1 for `precommit.json`). The save format is Parquet — efficient, single-file per object, dtype-safe.

2. **Submit EQ#2.** Copy your `overall` verdict and rationale from `integration_gate.json` into Notion under EQ#2 for this notebook.

The cell below writes the four data files, then reads each one back and confirms shapes and dtypes match what was in memory. Read-back is the load-bearing check: if a Parquet write silently truncates or coerces a dtype, the diagnostic surfaces here rather than three sections into Notebook 02.

You've integrated AtGenExpress and Wang 2023, certified the result through the integration gate, and have an `integration_gate.json` that records the verdict. Two things remain:

1. **Save the integrated dataset to disk.** Notebook 02 will load these files in its own Section 1 (defensive load, same pattern as this notebook's Section 1 for `precommit.json`). The save format is Parquet — efficient, single-file per object, dtype-safe.

2. **Submit EQ#2.** Copy your `overall` verdict and rationale from `integration_gate.json` into Notion under EQ#2 for this notebook.

The cell below writes the four data files, then reads each one back and confirms shapes and dtypes match what was in memory. Read-back is the load-bearing check: if a Parquet write silently truncates or coerces a dtype, the diagnostic surfaces here rather than three sections into Notebook 02.

In [ ]:
import pandas as pd
import json
from pathlib import Path

# Four data files plus the gate file already on disk from Section 8.
outputs = {
    "expr_shoot_combat.parquet": expr_shoot_combat,
    "expr_wang_combat.parquet":  expr_wang_combat,
    "meta_shoot.parquet":        meta_shoot,
    "meta_wang.parquet":         meta_wang,
}

# Write.
for filename, df in outputs.items():
    df.to_parquet(filename)
    print(f"Wrote {filename}")
print()

# Read back and confirm shape + dtype match.
for filename, original in outputs.items():
    reloaded = pd.read_parquet(filename)
    if reloaded.shape != original.shape:
        raise ValueError(
            f"{filename}: shape mismatch on read-back. "
            f"Original {original.shape}, reloaded {reloaded.shape}."
        )
    if not reloaded.columns.equals(original.columns):
        raise ValueError(f"{filename}: column mismatch on read-back.")
    if not reloaded.index.equals(original.index):
        raise ValueError(f"{filename}: index mismatch on read-back.")
    print(f"{filename}: {reloaded.shape[0]:,} x {reloaded.shape[1]:,} ✓")
print()

# Confirm integration_gate.json is in place and overall verdict is recorded.
with Path("integration_gate.json").open() as f:
    gate = json.load(f)
print(f"integration_gate.json: overall = {gate['overall']['verdict']!r}")
print()

# Confirm precommit.json is still present (NB02 reads it too).
if not Path("precommit.json").exists():
    raise FileNotFoundError("precommit.json is missing; Notebook 02 will fail to load.")
print("precommit.json: present ✓")
print()

print("Notebook directory now contains:")
for p in sorted(Path(".").glob("*.parquet")) + sorted(Path(".").glob("*.json")):
    print(f"  {p.name}")

**What you submit this week (EQ#2):**

Open `integration_gate.json` and copy the `overall.verdict` and `overall.rationale` values into Notion under EQ#2 for this notebook. That's the deliverable — your verdict on whether the integrated dataset is fit for downstream use, and the reasoning behind it.

**What Notebook 02 picks up:**

- `expr_shoot_combat.parquet`, `expr_wang_combat.parquet` — the batch-corrected expression matrices.
- `meta_shoot.parquet`, `meta_wang.parquet` — sample-level metadata, with `stress_status` attached as a column.
- `precommit.json` — re-read defensively in NB02's Section 1.
- `integration_gate.json` — read defensively in NB02's Section 1. NB02 will refuse to run if `overall.verdict == "fail"`.

Notebook 02 builds a stress-classifier on this integrated dataset and evaluates within-study accuracy. The classifier honors `meta_wang["ood"]` — samples tagged out-of-distribution (the 168 h timepoint, on the `"include_as_ood"` path) are held out from training and evaluated in Notebook 03.

If your overall verdict was `partial`, document in your EQ#2 rationale what you observed and why you chose to continue. NB02 will run on a `partial` gate but the constraint is on you to know what compromise you're carrying forward.